# Entity Graph (preview): Following Relationships Across Documents

An **entity graph** stores typed **entities** (people, missions, places, …) as nodes and their **relationships** as edges. It answers questions whose answer is a *chain of facts spread across different documents*, which ordinary retrieval struggles with.

In this notebook you will:

1. See **why** plain retrieval fails on a multi-hop question
2. Learn what an **ontology** is and design one around the questions you want to answer
3. Watch an LLM turn text into **triples** locally, to see what the service does at scale
4. Build a managed entity graph with a **graph-enabled Blob knowledge source** and an **ingestion cache**
5. **Inspect** the graph with GQL and draw it
6. **Evolve the ontology** from v1 to v2. Thanks to the cache, only the extraction step is redone
7. Query the graph through a **knowledge base** (natural language) and **directly with GQL**

![Entity graph overview: Wikipedia articles and an ontology feed LLM extraction into a graph index; changing the ontology replays the ingestion cache; the graph is queried through a knowledge base or directly with GQL](../docs/images/entity-graph-overview.svg)

## Prerequisites

| Requirement | Notes |
|---|---|
| [prepare-dataset.ipynb](prepare-dataset.ipynb) completed | Uploads the `space-entity` container and builds the `space-exploration` index used for the baseline |
| Search service in a region with knowledge graph support | |
| Storage account: **general-purpose v2** with Blob **and Table** endpoints, **not** ADLS Gen2 | Table storage is needed by the ingestion cache. Shared-key access can stay disabled |
| Search managed identity: **Storage Blob Data Contributor** and **Storage Table Data Contributor** on the storage account | Reads the source blobs and reads/writes the ingestion cache, keylessly |
| Blob **soft delete on**, blob **versioning off**, before the first indexing run | Notebook 00 turns soft delete on and checks versioning |
| Search managed identity: **Cognitive Services User** on the Foundry resource | Extraction and embedding calls during ingestion |
| A chat deployment for extraction (`GRAPH_LLM_DEPLOYMENT`, `gpt-5.6-luna` recommended) | Falls back to `LLM_DEPLOYMENT` |
| Your identity: **Search Service Contributor**, **Search Index Data Reader**; **Cognitive Services OpenAI User** on the Foundry resource | The last role is only for the local extraction demo in Step 3 |

> **Costs:** during preview the entity graph feature itself isn't charged, but extraction calls your chat model for every chunk. The `space-entity` container holds about 60 articles truncated to 12,000 characters each, to keep that bounded.
>
> **Limitations:** entity graph retrieval needs a **new** knowledge base (you can't enable it on an existing one). ADLS Gen2, ACL ingestion and sensitivity-label ingestion aren't supported. The feature is API-only; the [Foundry IQ skills plugin](../docs/foundry-iq-skills-guide.md) adds a guided UI.

## Setup

In [ ]:
# %pip install azure-search-documents==12.2.0b1 azure-identity azure-storage-blob python-dotenv requests networkx matplotlib

In [ ]:
import base64
import json
import os
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import requests
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    AzureBlobKnowledgeSource,
    AzureBlobKnowledgeSourceParameters,
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceGraphExtractionParameters,
    KnowledgeSourceGraphOntology,
    KnowledgeSourceReference,
    KnowledgeSourceResultsProcessing,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    AgentRuntimeGraphHydrateDocumentsParams,
    AgentRuntimeGraphParams,
    AgentRuntimeGraphProvenanceParams,
    AzureBlobKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceCache,
    KnowledgeSourceIngestionParameters,
)
from azure.search.documents.models import (
    GraphQueryProvenanceHydrateDocumentsParameters,
    GraphQueryProvenanceParameters,
    QueryType,
    VectorizableTextQuery,
)
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

load_dotenv(override=True)  # pick up .env edits without restarting the kernel

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_api_key = os.environ.get("AZURE_SEARCH_API_KEY")
credential = DefaultAzureCredential()
search_credential = AzureKeyCredential(search_api_key) if search_api_key else credential

ai_services_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].split("/api/projects")[0]
graph_llm_deployment = os.environ.get("GRAPH_LLM_DEPLOYMENT") or os.environ["LLM_DEPLOYMENT"]
graph_llm_model = os.environ.get("GRAPH_LLM_MODEL") or os.environ["LLM_MODEL"]
embedding_deployment = os.environ["EMBEDDING_DEPLOYMENT"]
embedding_model = os.environ["EMBEDDING_MODEL"]
# The search service reaches storage with its managed identity; no account key is used.
storage_resource_id = os.environ["AZURE_STORAGE_RESOURCE_ID"]
storage_search_connection = f"ResourceId={storage_resource_id};"
storage_account_url = f"https://{storage_resource_id.rstrip('/').split('/')[-1]}.blob.core.windows.net"

DATA_DIR = Path("..") / "data" / "space-exploration"
CONTAINER_ENTITY = "space-entity"
BASELINE_INDEX = "space-exploration"
KNOWLEDGE_SOURCE = "space-entity-ks"
KNOWLEDGE_BASE = "space-entity-kb"
GRAPH_NAME = "space"

index_client = SearchIndexClient(search_endpoint, search_credential)
indexer_client = SearchIndexerClient(search_endpoint, search_credential)

print(f"Extraction model: {graph_llm_deployment} / {graph_llm_model}")
print(f"Search endpoint:  {search_endpoint}")

## Step 1: Why do we need a graph?

Take this question:

> *Which astronauts flew on missions that launched from a site in Kazakhstan?*

The answer is a **chain**: *astronaut → crewed → mission → launched from → launch site → located in → country*. Each link usually lives in a **different article**: Yuri Gagarin's page says he flew Vostok 1, the Vostok 1 page says it launched from Baikonur, and the Baikonur page says it's in Kazakhstan.

Vector or keyword search scores each chunk **independently** against the question. It finds chunks that *mention* Kazakhstan or launch sites, but nothing joins "Gagarin" to "Kazakhstan" through the intermediate mission. Let's see that on the index from notebook 00:

In [ ]:
MULTI_HOP_QUESTION = "Which astronauts flew on missions that launched from a site in Kazakhstan?"

baseline_client = SearchClient(search_endpoint, BASELINE_INDEX, search_credential)
baseline = baseline_client.search(
    search_text=MULTI_HOP_QUESTION,
    vector_queries=[VectorizableTextQuery(text=MULTI_HOP_QUESTION, k_nearest_neighbors=50, fields="text_vector")],
    query_type=QueryType.SEMANTIC,
    select=["title", "chunk"],
    top=8,
)
for doc in baseline:
    print(f"{doc['title']:<28} {' '.join(doc['chunk'].split())[:110]}…")


The results are dominated by the **Baikonur Cosmodrome** and **Kazakhstan**-related chunks. They describe the place, not *who* flew from it, so an LLM grounded on them can only guess at a partial list. An entity graph stores the links explicitly, so the chain can be **traversed** instead of hoped for.

## Step 2: Design the ontology

An **ontology** tells the extractor *what to look for*: which **entity types** (classes) exist and which **relationships** (properties) may connect them. Foundry IQ takes a constrained **JSON-LD** document:

| JSON-LD element | Meaning | Example |
|---|---|---|
| `owl:Class` | An entity type; becomes a node label | `ex:Astronaut` |
| `owl:ObjectProperty` | A relationship type; becomes an edge predicate | `ex:crewOf` |
| `rdfs:domain` / `rdfs:range` | The allowed source and target types of a relationship | `Astronaut` → `Mission` |
| `rdfs:comment` | Natural-language guidance for the extraction model | "The astronaut flew as a crew member…" |

**Design the ontology from the questions you need to answer.** Version 1 deliberately covers only people, missions and agencies, so you can see what it *can't* answer before we extend it.

In [ ]:
def describe(ontology: dict) -> None:
    for item in ontology["@graph"]:
        name = item["@id"].split(":")[-1]
        if item["@type"] == "owl:Class":
            print(f"  class     {name:<14} {item['rdfs:comment']}")
        else:
            domain, range_ = item["rdfs:domain"]["@id"].split(":")[-1], item["rdfs:range"]["@id"].split(":")[-1]
            print(f"  relation  {name:<14} {domain} → {range_}")


def classes(ontology: dict) -> list[str]:
    return [i["@id"].split(":")[-1] for i in ontology["@graph"] if i["@type"] == "owl:Class"]

In [ ]:
ontology_v1 = json.loads((DATA_DIR / "ontology.v1.json").read_text(encoding="utf-8"))
ontology_v2 = json.loads((DATA_DIR / "ontology.v2.json").read_text(encoding="utf-8"))

print("Ontology v1:")
describe(ontology_v1)
added = {i["@id"] for i in ontology_v2["@graph"]} - {i["@id"] for i in ontology_v1["@graph"]}
print(f"\nOntology v2 adds: {sorted(a.split(':')[-1] for a in added)}")

## Step 3: How a graph is generated (local demo)

Graph extraction is, at its core, an LLM reading each chunk and emitting **triples** `(subject, predicate, object)` that fit the ontology. Entities that appear in many chunks are merged into a single node, and every triple keeps a pointer back to its source chunk (**provenance**).

The helper below calls your chat deployment directly. This is only an illustration of the idea; the service's own extraction prompt, entity merging and provenance tracking are managed for you in Step 4.

In [ ]:
def extract_triples(text: str, ontology: dict) -> list[dict]:
    """Ask the chat deployment for ontology-constrained triples (illustration only)."""
    token = credential.get_token("https://cognitiveservices.azure.com/.default").token
    prompt = (
        "Extract knowledge-graph triples from the text. Use ONLY the classes and relationships in this ontology; "
        "ignore facts that don't fit it. Return JSON: {\"triples\": [{\"subject\", \"subjectType\", \"predicate\", "
        "\"object\", \"objectType\"}]}.\n\nOntology:\n" + json.dumps(ontology["@graph"]) + "\n\nText:\n" + text
    )
    response = requests.post(
        f"{ai_services_endpoint}/openai/deployments/{graph_llm_deployment}/chat/completions?api-version=2024-10-21",
        headers={"Authorization": f"Bearer {token}"},
        json={"messages": [{"role": "user", "content": prompt}], "response_format": {"type": "json_object"}},
        timeout=120,
    )
    response.raise_for_status()
    return json.loads(response.json()["choices"][0]["message"]["content"])["triples"]

In [ ]:
SAMPLE_TEXT = """\
Yuri Gagarin, a pilot selected by the Soviet space program, became the first human in space aboard Vostok 1 in April 1961.
Vostok 1 lifted off from the Baikonur Cosmodrome, the launch site that is located in Kazakhstan.
"""


def short(term: str) -> str:
    return term.split(":")[-1]


for label, ontology in (("v1", ontology_v1), ("v2", ontology_v2)):
    print(f"--- Triples with ontology {label} ---")
    for t in extract_triples(SAMPLE_TEXT, ontology):
        print(f"  ({t['subject']} : {short(t['subjectType'])}) -[{short(t['predicate'])}]-> ({t['object']} : {short(t['objectType'])})")


With **v1** the launch-site and country facts are dropped: the ontology has no types for them. With **v2** you get the missing links: `Vostok 1 -[launchedFrom]-> Baikonur Cosmodrome -[locatedIn]-> Kazakhstan`. **The ontology determines the shape of the graph.**

## Step 4: Build the managed entity graph

A **graph-enabled Blob knowledge source** does everything for you: it generates a data source, skillset, indexer, a **chunk index** and a **graph index**, and runs extraction during ingestion.

Two settings matter for this walkthrough:

**`graph_extraction`** names the graph and supplies the ontology. `ontology.value` is the JSON-LD document **serialized as a string**, and `page_degree_of_parallelism` controls how many chunks are extracted concurrently.

**`ingestion_parameters.cache`** is the **ingestion cache**. It stores intermediate enrichment results (cracked text, chunks, embeddings) in your storage account, so that when the ontology changes the service can **replay** the cache and rerun only graph extraction instead of reprocessing every document. Cache rules:

- It must be configured **at creation**. You can't add it later, remove it, or change its storage binding
- `enable_reprocessing` must be `True` for graph extraction cache replay
- The storage account must be general-purpose v2 with **Blob and Table** endpoints
- Without a cache, the ontology can't be changed after creation

> Both the source and the cache use a **managed-identity** connection (`ResourceId=/subscriptions/.../storageAccounts/<account>;`), so no account key is needed and shared-key access can stay disabled. Because no `identity` is set, the search service's **system-assigned** identity is used. It needs **Storage Blob Data Contributor** (source reads plus cache blobs) and **Storage Table Data Contributor** (cache tables) on the account.

### Knowledge source definition

In [ ]:
def build_knowledge_source(ontology: dict, ontology_name: str) -> AzureBlobKnowledgeSource:
    def aoai(deployment: str, model: str) -> AzureOpenAIVectorizerParameters:
        return AzureOpenAIVectorizerParameters(resource_url=ai_services_endpoint, deployment_name=deployment, model_name=model)

    return AzureBlobKnowledgeSource(
        name=KNOWLEDGE_SOURCE,
        description="Astronauts, missions, agencies, launch sites and places from Wikipedia.",
        results_processing=KnowledgeSourceResultsProcessing.NONE,
        azure_blob_parameters=AzureBlobKnowledgeSourceParameters(
            connection_string=storage_search_connection,
            container_name=CONTAINER_ENTITY,
            is_adls_gen2=False,
            ingestion_parameters=KnowledgeSourceIngestionParameters(
                content_extraction_mode="minimal",
                disable_image_verbalization=True,
                chat_completion_model=KnowledgeBaseAzureOpenAIModel(
                    azure_open_ai_parameters=aoai(graph_llm_deployment, graph_llm_model)),
                embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                    azure_open_ai_parameters=aoai(embedding_deployment, embedding_model)),
                cache=KnowledgeSourceCache(storage_connection_string=storage_search_connection, enable_reprocessing=True),
            ),
            graph_extraction=KnowledgeSourceGraphExtractionParameters(
                graph_name=GRAPH_NAME,
                page_degree_of_parallelism=4,
                ontology=KnowledgeSourceGraphOntology(
                    kind="inline", name=ontology_name, value=json.dumps(ontology, separators=(",", ":"))),
            ),
        ),
    )

### Synchronization helpers

Extraction is complete when `last_synchronization_state.end_time` is set and `current_synchronization_state` is empty. `synchronization_status == "active"` only means the source is *available* for synchronization, not that it's finished. `graph_statistics.triples_written` reports how many relationships that synchronization wrote.

`apply_ontology` is used for both the first build and the later ontology change. A **new** knowledge source starts ingesting on its own; an **updated** one doesn't always, so if nothing starts within two minutes it runs the generated indexer once. That also makes this notebook safe to re-run from the top.


In [ ]:
def wait_for_sync(not_before: float, poll_seconds: int = 30):
    """Poll the knowledge source until a synchronization that ended after `not_before` is reported."""
    while True:
        status = index_client.get_knowledge_source_status(KNOWLEDGE_SOURCE)
        current, last = status.current_synchronization_state, status.last_synchronization_state
        if current:
            print(f"{time.strftime('%H:%M:%S')}  running: {current.items_updates_processed} documents processed, {current.items_updates_failed} failed")
            for error in (current.errors or [])[:3]:
                print(f"    error: {error}")
        elif last and last.end_time and last.end_time.timestamp() >= not_before:
            print(f"{time.strftime('%H:%M:%S')}  finished: {last.items_updates_processed} documents processed, {last.items_updates_failed} failed")
            if last.graph_statistics:
                print(f"Relationships (triples) written to the graph: {last.graph_statistics.triples_written}")
            return status
        else:
            print(f"{time.strftime('%H:%M:%S')}  waiting for synchronization to start…")
        time.sleep(poll_seconds)


def apply_ontology(ontology: dict, ontology_name: str):
    """Create or update the knowledge source with an ontology and wait until extraction finishes."""
    if KNOWLEDGE_SOURCE in {ks.name for ks in index_client.list_knowledge_sources()}:
        assert not index_client.get_knowledge_source_status(KNOWLEDGE_SOURCE).current_synchronization_state, \
            "A synchronization is still running; wait for it to finish before changing the ontology."

    started = time.time()
    index_client.create_or_update_knowledge_source(build_knowledge_source(ontology, ontology_name))
    print(f"Knowledge source '{KNOWLEDGE_SOURCE}' now uses ontology {ontology_name}")

    # A new source starts ingesting on its own; an updated one may need its generated indexer run once.
    time.sleep(120)
    if not index_client.get_knowledge_source_status(KNOWLEDGE_SOURCE).current_synchronization_state:
        indexer = dict(index_client.get_knowledge_source(KNOWLEDGE_SOURCE).azure_blob_parameters.created_resources or {}).get("indexer")
        if indexer:
            indexer_client.run_indexer(indexer)
            print(f"No synchronization started automatically; ran '{indexer}'")
    return wait_for_sync(started)


### Create the source with ontology v1 and wait

In [ ]:
status_v1 = apply_ontology(ontology_v1, "space-v1")


### See what was generated

The knowledge source owns everything it generated. Read the definition back to find their names: `graphIndex` is the graph you'll query, and `index` holds the chunks used for provenance. Don't edit these objects directly.

In [ ]:
definition = index_client.get_knowledge_source(KNOWLEDGE_SOURCE)
created = dict(definition.azure_blob_parameters.created_resources or {})
print(json.dumps(created, indent=2))

GRAPH_INDEX = created["graphIndex"]
GENERATED_INDEXER = created.get("indexer")
# In 12.2.0b1, ingestion_parameters comes back as raw JSON (a dict), not a model.
cache = definition.azure_blob_parameters.ingestion_parameters["cache"]
print(f"\nCache configured: enableReprocessing={cache['enableReprocessing']}")


## Step 5: Inspect the graph with GQL

The graph index answers **GQL** (Graph Query Language) through `SearchClient.graph`. The building blocks:

- `MATCH (a:Astronaut)-[r]->(m:Mission)`: a pattern of labelled nodes and edges
- `r.predicate`: the relationship name from the ontology (for example `crewOf`)
- `n.name`: the entity's name
- `RETURN … LIMIT n`: the columns to return; each row behaves like a dictionary keyed by these aliases

### Graph helpers

In [ ]:
graph_client = SearchClient(search_endpoint, GRAPH_INDEX, search_credential)


def gql(query: str, provenance: bool = False):
    """Run a GQL query on the managed graph; optionally hydrate the supporting source chunks."""
    provenance_params = GraphQueryProvenanceParameters(
        kind="hydrate", max_evidence_per_row=3,
        hydrate_documents=GraphQueryProvenanceHydrateDocumentsParameters(select="uid,snippet,snippet_parent_id,blob_url"),
    ) if provenance else None
    return graph_client.graph(gql=query, include_statistics=True, provenance=provenance_params).value or []


def load_graph(ontology: dict) -> nx.DiGraph:
    """Pull every node and outgoing edge, one ontology class at a time, into a networkx graph."""
    graph = nx.DiGraph()
    for label in classes(ontology):
        for row in gql(f"MATCH (n:{label}) RETURN n.name AS name LIMIT 2000"):
            graph.add_node(row["name"], label=label)
    # The service rejects unlabeled full-graph scans, so anchor each edge query on a class.
    for label in classes(ontology):
        for row in gql(f"MATCH (a:{label})-[r]->(b) RETURN a.name AS source, r.predicate AS predicate, b.name AS target LIMIT 5000"):
            graph.add_edge(row["source"], row["target"], predicate=row["predicate"])
    return graph


def summarize(graph: nx.DiGraph) -> None:
    print(f"Entities: {graph.number_of_nodes()}, relationships: {graph.number_of_edges()}")
    print("Entities per class:      ", dict(Counter(d.get("label", "?") for _, d in graph.nodes(data=True))))
    print("Relationships per type:  ", dict(Counter(d["predicate"] for *_, d in graph.edges(data=True))))


def print_table(rows: list[dict], columns: list[str], max_width: int = 30) -> None:
    """Fixed-width table; only the last column is allowed to run long."""
    widths = {c: min(max_width, max([len(c)] + [len(str(r.get(c, ""))) for r in rows])) for c in columns[:-1]}
    cell = lambda r, c: str(r.get(c, ""))[:max_width].ljust(widths[c]) if c in widths else str(r.get(c, ""))
    print("  ".join(c.ljust(widths.get(c, 0)) for c in columns))
    print("  ".join("-" * widths.get(c, len(c)) for c in columns))
    for r in rows:
        print("  ".join(cell(r, c) for c in columns))
    print(f"({len(rows)} rows)")


def slugify(title: str) -> str:
    """Same blob naming as notebook 00."""
    ascii_title = unicodedata.normalize("NFKD", title).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Za-z0-9]+", "-", ascii_title).strip("-").lower()


ARTICLE_TITLES = {
    slugify(line.strip()): line.strip()
    for line in (DATA_DIR / "entity-subset.txt").read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.startswith("#")
}


def article_title(blob_url: str) -> str:
    slug = blob_url.rstrip("/").split("/")[-1].removesuffix(".txt")
    return ARTICLE_TITLES.get(slug, slug)


def decode_parent_id(parent_id: str) -> str:
    """Chunk parent ids are the blob URL, base64url-encoded with the padding count appended as a digit."""
    return base64.urlsafe_b64decode(parent_id[:-1] + "=" * int(parent_id[-1])).decode()


def source_articles(row) -> list[str]:
    evidence = (row.provenance or {}).get("evidence", [])
    return sorted({article_title(decode_parent_id(e["parentId"])) for e in evidence if e.get("parentId")})


COLORS = {"Astronaut": "#4c9be8", "Mission": "#f2a541", "Agency": "#7bc47f",
          "LaunchSite": "#d95f5f", "Place": "#a37acc", "Spacecraft": "#8c8c8c"}


def draw(graph: nx.DiGraph, title: str, center: str | None = None, radius: int = 2) -> None:
    """Draw the graph, or only the neighbourhood of the first node whose name contains `center`."""
    if center:
        match = next((n for n in graph if center.lower() in n.lower()), None)
        if match is None:
            print(f"No node matching '{center}'")
            return
        graph = nx.ego_graph(graph.to_undirected(as_view=True), match, radius=radius).copy()
    plt.figure(figsize=(14, 10))
    layout = nx.spring_layout(graph, k=0.6, seed=7)
    colors = [COLORS.get(graph.nodes[n].get("label"), "#cccccc") for n in graph]
    nx.draw_networkx(graph, layout, node_color=colors, node_size=500, font_size=7, arrows=True, edge_color="#bbbbbb")
    nx.draw_networkx_edge_labels(graph, layout, edge_labels=nx.get_edge_attributes(graph, "predicate"), font_size=6)
    plt.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=c, label=l) for l, c in COLORS.items()], loc="lower left")
    plt.title(title)
    plt.axis("off")
    plt.show()


### Load and draw the v1 graph

In [ ]:
graph_v1 = load_graph(ontology_v1)
summarize(graph_v1)

In [ ]:
draw(graph_v1, "Entity graph — ontology v1 (around Yuri Gagarin)", center="Gagarin")

### Query v1

A question **inside** v1's scope works with a two-hop traversal: *astronaut → mission → agency*.

In [ ]:
rows = gql(
    "MATCH (a:Astronaut)-[c]->(m:Mission)-[o]->(g:Agency) "
    "WHERE c.predicate = 'crewOf' AND o.predicate = 'operatedBy' "
    "RETURN a.name AS astronaut, m.name AS mission, g.name AS agency LIMIT 25"
)
print_table(rows, ["astronaut", "mission", "agency"])


The Kazakhstan question is **out of scope** for v1: there is no `LaunchSite` or `Place` type and no `launchedFrom` / `locatedIn` relationship, so the facts were never extracted.

In [ ]:
v1_predicates = {d["predicate"] for *_, d in graph_v1.edges(data=True)}
print("Predicates in v1 graph:", sorted(v1_predicates))
print("Can v1 express 'launchedFrom'?", "launchedFrom" in v1_predicates)

### Ask in natural language with the search SDK

GQL needs you to know the schema. A **knowledge base** lets you ask in plain language: it translates the question into GQL with its own model, runs it on the graph, and returns the result through `KnowledgeBaseRetrievalClient.retrieve`. Its model setting is **separate** from the extraction model, so configure it explicitly. `low` reasoning effort with `gpt-5.6-luna` is the recommended balance of quality and cost.

We create it now, while the graph still uses ontology v1, so we can ask the same question before and after the ontology change.


In [ ]:
knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE,
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE)],
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
        resource_url=ai_services_endpoint, deployment_name=graph_llm_deployment, model_name=graph_llm_model,
    ))],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    retrieval_instructions=(
        "Use graph_search for entity relationship questions. Answer relationship questions by traversing "
        "the graph rather than using ordinary text search."
    ),
)
index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE}' created or updated")


#### Knowledge base helpers

- `ask` sends the question. `EXTRACTIVE_DATA` returns the raw graph rows; `ANSWER_SYNTHESIS` returns a written, cited answer.
- `show_answer` prints the steps the knowledge base took, the GQL it generated, and either the rows as a table or the written answer.
- `show_references` lists the supporting Wikipedia passages.
- `compare_answers` prints the generated GQL and answer for several responses, one after another.


In [ ]:
kb_client = KnowledgeBaseRetrievalClient(search_endpoint, search_credential, knowledge_base_name=KNOWLEDGE_BASE)


def ask(question: str, output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA):
    request = KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=question)])],
        include_activity=True,
        output_mode=output_mode,
        max_runtime_in_seconds=60,
        knowledge_source_params=[AzureBlobKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE,
            include_references=True,
            include_reference_source_data=True,
            fail_on_error=True,
            graph_parameters=AgentRuntimeGraphParams(
                include_query=True,
                provenance=AgentRuntimeGraphProvenanceParams(
                    kind="hydrate",
                    hydrate_documents=AgentRuntimeGraphHydrateDocumentsParams(select="uid,snippet,snippet_parent_id,blob_url", top=10),
                ),
            ),
        )],
    )
    return kb_client.retrieve(request)


def graph_searches(response) -> list:
    return [r.graph_search_arguments for r in response.activity or [] if getattr(r, "graph_search_arguments", None)]


def graph_rows(response) -> tuple[list[str], list[dict]]:
    """The graph result is the reference whose source data has columns and rows."""
    for ref in response.references or []:
        data = ref.source_data or {}
        if "rows" in data:
            return data.get("columns", []), data["rows"]
    return [], []


def show_activity(response) -> None:
    print("What the knowledge base did:")
    for record in response.activity or []:
        elapsed = f"{record.elapsed_ms} ms" if record.elapsed_ms else ""
        print(f"  {record.type:<22} {elapsed}")
    for search in graph_searches(response):
        print(f"\nGenerated GQL ({search.outcome}):\n  {search.gql}")


def show_answer(response) -> None:
    show_activity(response)
    columns, rows = graph_rows(response)
    text = response.response[0].content[0].text
    if text.lstrip().startswith("["):  # extractive mode returns the grounding as JSON
        print("\nGraph rows returned:")
        print_table(rows, columns) if rows else print("  (none)")
    else:
        print(f"\nAnswer:\n{text}")


def best_sentence(snippet: str, names: set[str]) -> str:
    """The sentence in a passage that mentions the most entities from the result rows."""
    sentences = re.split(r"(?<=[.!?])\s+", " ".join(snippet.split()))
    return max(sentences, key=lambda s: sum(n in s for n in names))[:300]


def show_references(response, limit: int = 8) -> None:
    columns, rows = graph_rows(response)
    names = {str(v) for r in rows for v in r.values()}
    passages = [ref.source_data for ref in response.references or [] if (ref.source_data or {}).get("snippet")]
    print(f"Graph result: {len(rows)} rows. Supporting passages ({len(passages)}):\n")
    for data in passages[:limit]:
        print(f"- {article_title(data['blob_url'])}")
        print(f"  \"{best_sentence(data['snippet'], names)}\"")


def compare_answers(answers: dict) -> None:
    """Print the generated GQL and the written answer for each labelled response."""
    for label, response in answers.items():
        print(f"{'=' * 25} {label} {'=' * 25}")
        searches = graph_searches(response)
        for search in searches:
            print(f"Generated GQL ({search.outcome}):\n  {search.gql}")
        if not searches:
            print("Generated GQL: none")
        print(f"\nAnswer:\n{response.response[0].content[0].text}\n")


#### Ask the multi-hop question against the v1 graph

`ANSWER_SYNTHESIS` returns a written answer instead of raw rows, which makes the v1 → v2 difference easy to read. With v1 the knowledge base can only generate *astronaut → mission* GQL (there is no launch-site relationship to traverse), so expect a hedged, partial answer that guesses from text. We'll ask the exact same question again after the ontology change in Step 7b.


In [ ]:
answer_v1 = ask(MULTI_HOP_QUESTION, KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS)
show_answer(answer_v1)


## Step 6: Evolve the ontology (v1 → v2) using the cache

Ontology v2 adds `LaunchSite`, `Place` and `Spacecraft`, with `launchedFrom`, `locatedIn` and `usedSpacecraft`.

To change it, update the knowledge source with the new ontology. Rules:

- Wait until **no synchronization is active**. Ontology changes are rejected during a sync
- An attached ontology can be **replaced but not removed**

Because the source has an ingestion cache with `enable_reprocessing`, the service **replays cached enrichment** (cracking, chunking, embeddings) and reruns only graph extraction with the new ontology. Without the cache this change wouldn't be allowed at all.


In [ ]:
status_v2 = apply_ontology(ontology_v2, "space-v2")


### Compare v1 and v2

Compare the entity and relationship counts, and look at the new relationship types.


In [ ]:
graph_v2 = load_graph(ontology_v2)

for label, graph in (("v1", graph_v1), ("v2", graph_v2)):
    print(f"--- {label}")
    summarize(graph)

In [ ]:
draw(graph_v2, "Entity graph — ontology v2 (around Baikonur)", center="Baikonur")

## Step 7: Ask the multi-hop question

### 7a. Directly with GQL

A **three-hop** traversal: *astronaut → mission → launch site → place*. With provenance on, each row also lists the Wikipedia articles whose passages support it.


In [ ]:
rows = gql(
    "MATCH (a:Astronaut)-[c]->(m:Mission)-[l]->(s:LaunchSite)-[p]->(place:Place) "
    "WHERE c.predicate = 'crewOf' AND l.predicate = 'launchedFrom' AND p.predicate = 'locatedIn' "
    "AND place.name = 'Kazakhstan' "
    "RETURN a.name AS astronaut, m.name AS mission, s.name AS site, place.name AS place LIMIT 50",
    provenance=True,
)
table = [{**{k: row[k] for k in ("astronaut", "mission", "site", "place")},
          "evidence from": ", ".join(source_articles(row))} for row in rows]
print_table(table, ["astronaut", "mission", "site", "place", "evidence from"])


How to read this:

- **Evidence covers the whole path**, not just the astronaut. Every row shares the *Baikonur Cosmodrome → Kazakhstan* link, so articles that state that fact (for example *Tim Peake* or *Vostochny Cosmodrome*) appear on many rows. Only the first 3 passages per row are returned (`max_evidence_per_row=3`)
- **Entity naming isn't merged**: "Gagarin" and "Yuri Gagarin" are separate nodes
- **Extraction can be wrong**: Alexei Leonov and Valery Kubasov were the *original* Soyuz 11 crew and were replaced before launch, but the extractor recorded them as `crewOf`. Always spot-check rows against their evidence

If the result is empty, check how places and launch sites were named during extraction (for example `Kazakhstan` vs `Kazakh SSR`):


In [ ]:
for label in ("Place", "LaunchSite"):
    print(f"{label}: {sorted(n for n, d in graph_v2.nodes(data=True) if d.get('label') == label)}")

### 7b. Through the knowledge base: v1 vs v2

Ask the **same natural-language question** through the same knowledge base that answered it in Step 5. Nothing about the knowledge base changed; only the ontology behind the graph did. Compare the generated GQL and the written answers side by side.


In [ ]:
answer_v2 = ask(MULTI_HOP_QUESTION, KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS)
compare_answers({"ontology v1": answer_v1, "ontology v2": answer_v2})


#### Raw rows for your own app

An agent or app usually wants the **data**, not prose. `EXTRACTIVE_DATA` (the knowledge base default) returns the graph rows as JSON, ready to be passed to your own model.


In [ ]:
answer = ask(MULTI_HOP_QUESTION + " Return astronaut, mission, launch site and country.")
show_answer(answer)


#### Where the answer came from

Besides the graph rows, provenance hydration returns the **source passages** behind the relationships. For each one, the helper shows the Wikipedia article and the sentence that mentions the most entities from the result, so you can check a row against its source.


In [ ]:
show_references(answer)

## Validate before you scale

- **Start from questions.** Write 5–10 representative questions and check that each maps to a path through the ontology
- **Check extraction quality** with direct GQL before blaming the knowledge base: an empty KB answer doesn't prove extraction failed, and vice versa
- **Watch entity naming.** Variants such as "Baikonur" vs "Baikonur Cosmodrome" split nodes. For stricter entity resolution, `graph_extraction.grounding` can resolve extracted entities against a reference search index on the same service
- **Estimate cost** from corpus size *and* the number of extracted entities and relationships
- **Throttling:** a `502` with *All retrieval tasks failed* often means the knowledge base's model returned `429`. Check that deployment's quota

## Clean up

Delete the knowledge base first, then the knowledge source. Deleting the source removes its generated data source, skillset, indexer, chunk index and graph index, but **not** your blobs. The last cell lists any enrichment-cache containers left in the storage account so you can decide whether to delete them.

In [ ]:
# index_client.delete_knowledge_base(KNOWLEDGE_BASE)
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE)
# print("Deleted knowledge base and knowledge source")

In [ ]:
blob_service = BlobServiceClient(storage_account_url, credential=credential)
cache_containers = [c.name for c in blob_service.list_containers() if "cache" in c.name]
print("Containers that look like enrichment caches:", cache_containers or "none")
# for name in cache_containers:
#     blob_service.delete_container(name)

## Summary

| Concept | What you saw |
|---|---|
| Why a graph | Retrieval scores chunks independently; a graph stores the links so multi-hop questions can be traversed |
| Ontology | JSON-LD classes and relationships, designed from target questions; it determines what gets extracted |
| Extraction | An LLM turns each chunk into triples with provenance, managed by a graph-enabled Blob knowledge source |
| Ingestion cache | Must be set at creation; lets you change the ontology and replay cached enrichment instead of reprocessing everything |
| Querying | Natural language through a knowledge base (generated GQL), or direct GQL with `SearchClient.graph` |

### Next steps

- [topic-graph.ipynb](topic-graph.ipynb): theme-level questions with a topic graph
- [docs/foundry-iq-skills-guide.md](../docs/foundry-iq-skills-guide.md): build and explore entity graphs with the Foundry IQ skills plugin